# U8 — Data Cleaning & Preprocessing (Part 1): Lab

**Cleaning Messy Data** — profiling · missing values · duplicates · data types · outliers · messy text

_Day 4 · Phase C — Data Engineering & Preparation · The highest-leverage work in ML: clean data first._

#objectives

By the end of this lab you will be able to:

Profile a dataset to find quality problems before fixing anything

Detect missing values — including disguised ones — and handle them by dropping or imputing

Remove duplicate rows and fix wrong data types

Detect outliers with the IQR rule and decide how to treat them

Standardise messy text so one real category isn't split into many

#how to use this lab

Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each `# YOUR CODE HERE` with working code.

Run cells with **Shift + Enter**. Run the demos before attempting the exercises.

In [ ]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

In [ ]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

#1. Profile the data — find the problems

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts

In [ ]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

#### 🧪 LAB EXERCISE 1 — Profile it yourself

1. Print the number of **duplicate rows**.
2. Print the **missing count per column**.
3. In a comment, list at least **three** problems you can already see in the data.

In [ ]:
# 1. duplicate row count
# YOUR CODE HERE

# 2. missing per column
# YOUR CODE HERE

# 3. Problems I can see: ...   (write 3+ in this comment)

#2. Missing values — detect & handle

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

In [ ]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

#### 🧪 LAB EXERCISE 2 — Compare drop vs impute

Start from a fresh messy copy (`ex = raw.copy()`):
1. Convert `spend` to numeric and `age`'s `-1` to NaN (as in 2A).
2. Make a **dropna** version and a **median-impute** version.
3. Print the row count of each — how many rows did dropping cost you?

In [ ]:
ex = raw.copy()

# 1. unmask missing values (spend -> numeric, age -1 -> NaN)
# YOUR CODE HERE

# 2a. dropna version
# YOUR CODE HERE

# 2b. median-impute version
# YOUR CODE HERE

# 3. compare row counts
# YOUR CODE HERE

#3. Duplicates & data types

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

In [ ]:
# -----------------------------------------------------------
# 🔹 3B. FIX DATA TYPES
# -----------------------------------------------------------

# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)

#### 🧪 LAB EXERCISE 3 — Dedupe & retype

On a fresh copy `ex = raw.copy()`:
1. Convert `spend` to numeric and `date` to datetime.
2. Drop duplicate rows.
3. Print `.dtypes` and the final `.shape`.

In [ ]:
ex = raw.copy()

# 1. fix types: spend -> numeric, date -> datetime
# YOUR CODE HERE

# 2. drop duplicates
# YOUR CODE HERE

# 3. dtypes + shape
# YOUR CODE HERE

#4. Outliers — detect with the IQR rule

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

In [ ]:
# -----------------------------------------------------------
# 🔹 4B. ONE WAY TO TREAT THEM — CAP (winsorise)
# -----------------------------------------------------------

# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

#### 🧪 LAB EXERCISE 4 — Find the outliers

Using the `age` column of the cleaned `df`:
1. Compute Q1, Q3 and the IQR.
2. Compute the lower & upper bounds (`Q1 − 1.5·IQR`, `Q3 + 1.5·IQR`).
3. Print any rows whose `age` falls outside those bounds (there may be none — that's a valid result).

In [ ]:
# 1. Q1, Q3, IQR for 'age'
# YOUR CODE HERE

# 2. lower & upper bounds
# YOUR CODE HERE

# 3. rows outside the bounds
# YOUR CODE HERE

#5. Messy text & inconsistent categories

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
# -----------------------------------------------------------

print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!

In [ ]:
# -----------------------------------------------------------
# 🔹 5B. STANDARDISE THE STRINGS
# -----------------------------------------------------------

s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories

#### 🧪 LAB EXERCISE 5 — Clean a messy column

The `messy` series below has the same problems.
1. Trim whitespace and lowercase it.
2. Map the variant `'n.y.'` to `'new york'`.
3. Print `value_counts()` — you should end up with just two clean categories.

In [ ]:
messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')

# 1. strip + lower
# YOUR CODE HERE

# 2. map 'n.y.' -> 'new york'  (after lowering)
# YOUR CODE HERE

# 3. value_counts()
# YOUR CODE HERE

#✅ The cleaned dataset

In [ ]:
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

#📘 Summary — Data-cleaning toolkit

| Step | What to check | Key calls |
| ---- | ------------- | --------- |
| **Profile** | overall health | `.info()` · `.isna().sum()` · `.describe()` · `.duplicated().sum()` |
| **Missing** | gaps & disguises | `pd.to_numeric(..., errors='coerce')` · `.replace(-1, np.nan)` · `.fillna()` |
| **Duplicates** | repeated rows | `.drop_duplicates()` |
| **Types** | text vs number/date | `pd.to_numeric` · `pd.to_datetime` · `.astype()` |
| **Outliers** | extreme values | IQR rule · `.clip(low, high)` |
| **Text** | inconsistent labels | `.str.strip()` · `.str.lower()` · `.replace({...})` |

**Homework (self-paced):** profile a messy CSV · unmask & impute missing values · drop duplicates · fix a numeric & a date column · flag IQR outliers · standardise a text column.

**Next — U8 Part 2:** preparing data for ML — encoding categoricals (label & one-hot), feature scaling (normalise & standardise), and feature-engineering basics.